# 13 - Synthetic records -> the lab's 524-number vectors (RAIM input) (run_02)

Runs the notebook 07 feature pipeline (whole-stay version, the one the lab's RAIM uses) on the synthetic Synthea files
from notebook 12, **without refitting anything**: the Word2Vec models, procedure-code map and category codes come from the
real `features_full` run, so synthetic vectors live in the same space as the real ones.

Outputs per synthetic set (`synthetic_full/seed42/<set>/features_524/`): `encounters_524.parquet`,
`sequences_patient_level.pkl` (same format RAIM reads; each synthetic patient has one stay), `features_meta.json`.

Checks:
1. **Coverage**: share of stays where each modality is present, and share of codes/words the real Word2Vec knows.
2. **Real vs synthetic in the 524-space**: discriminator AUC on all 524 numbers and per block, with the
   real-vs-real reference (training PLOS vs held-out PLOS).
3. **Does a model trained on real data see them as long stays?** A classifier trained on real `features_full` (train split)
   scores real held-out PLOS, real held-out non-PLOS and synthetic stays.

Cell 6 also prints the lab activity rate, real vs synthetic.
Runtime: CPU, about 5 minutes.

In [ ]:
# Cell 1 - settings
import os, sys, re, json, pickle, time, tarfile, numpy as np, pandas as pd
from datetime import timedelta
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
    import subprocess; subprocess.run([sys.executable, "-m", "pip", "-q", "install", "gensim"], check=True)
    BASE = "/content/drive/MyDrive/genai_synthea/run_02"
else:
    BASE = os.environ.get("RUN_BASE", "run_02")
REAL = f"{BASE}/features_full"                                   # real whole-stay features (notebook 07, WINDOW_HOURS = None)
SETS = {"matched": f"{BASE}/synthetic_full/seed42/matched_filtered",
        "tail_dense": f"{BASE}/synthetic_full/seed42/tail_dense_filtered"}
W2V_DIM, CHUNK = 128, 200_000
FM = json.load(open(f"{REAL}/features_meta.json"))
THR = FM["threshold"]
assert FM.get("window_hours") in (None, 0), "REAL must be the whole-stay run (features_full)"
if FM.get("llm_notes"): print("NOTE: the real run added LLM note codes to conditions; synthetic stays have none (small mismatch).")
if FM.get("scispacy"): print("NOTE: the real run used scispaCy abbreviation expansion; it is skipped here (synthetic notes have no abbreviations).")
from gensim.models import Word2Vec
MODS = ["conditions", "procedures", "notes", "medications"]
W2V = {m: Word2Vec.load(f"{REAL}/word2vec_{m}.model") for m in MODS}
PROC_MAP = pickle.load(open(f"{REAL}/desc_to_icd10pcs.pkl", "rb")) if os.path.exists(f"{REAL}/desc_to_icd10pcs.pkl") else {}
XR = pd.read_parquet(f"{REAL}/encounters_524.parquet")
RACE = dict(zip(XR.race, XR.race_enc)); ETH = dict(zip(XR.ethnicity, XR.ethnicity_enc))   # the real category codes
print(f"Real stays {len(XR):,} | Word2Vec vocab: " + ", ".join(f"{m} {len(W2V[m].wv):,}" for m in MODS) +
      f" | procedure map entries {len(PROC_MAP):,}")

In [ ]:
# Cell 2 - feature functions copied unchanged from notebook 07
import nltk
try: nltk.data.find("corpora/stopwords")
except LookupError: nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords
USE_SCISPACY = False
def expand(t): return t
FORM = ["TABLET", "CAPSULE", "INJECTION", "ORAL SOLUTION", "SOLUTION", "SUSPENSION", "SYRINGE", "VIAL", "PATCH",
        "BAG", "SPRAY", "CREAM", "OINTMENT", "SUPPOSITORY", "DROPPER", "DISKUS", "DEVICE", "LOZENGE", "TUBE",
        "ELIXIR", "LIQUID", "NEBULIZER", "ORAL", "POWDER", "GRANULES", "CONCENTRATE", "PREFILLED"]
def clean_drug_name(d):
    d = d.upper().strip(); d = re.sub(r'\*(NF|NON-FORMULARY|OB|DC)\*\s*', '', d)
    d = re.sub(r'\s*\(.*?\)', '', d); d = re.sub(r'\s*\d+(\.\d+)?\s*%$', '', d)
    for f in FORM: d = d.replace(f, '')
    return d.strip() or "UNKNOWN"
def dose(d):
    m = re.search(r'(\d+\.?\d*)\s*(mg|mcg|g|ml|iu|units|meq|%)', d, re.I)
    if m: return f"{m.group(1)}{m.group(2)}".upper()
    m = re.search(r'(\d+\.?\d*)\s*/\s*(\d+\.?\d*)\s*(mg|mcg|ml|units)', d, re.I)
    return f"{m.group(1)}/{m.group(2)}{m.group(3)}".upper() if m else "UNKNOWN"
def med_token(d):
    if not isinstance(d, str): return "UNKNOWN"
    n = re.sub(r'\d+\.?\d*\s*(MG|MCG|G|ML|IU|MEQ|UNITS)', '', clean_drug_name(d)).strip()
    n = re.sub(r'\s+', '_', n).strip('_')
    return f"{n}_{dose(d)}" if n and n != "UNKNOWN" else "UNKNOWN"

DATE_RE = re.compile(r'^\d{4}-\d{2}-\d{2}', re.M)
HEADER_RE = re.compile(r'^#{1,3}\s*(.+?)\s*$', re.M)
KEEP = {'hpi': r'\b(history of present illness|hpi)\b', 'allergies': r'\ballerg(y|ies)\b',
        'medications': r'\b(medications?|meds)\b', 'assessment_plan': r'\b(assessment|plan)\b',
        'radiology': r'\b(radiology|imaging)\b', 'nursing': r'\bnursing\b'}
KEEP = {k: re.compile(v, re.I) for k, v in KEEP.items()}
SOCIAL = re.compile(r'\bsocial history\b', re.I)

def segments(text):
    m = list(DATE_RE.finditer(text)); out = []
    for i, x in enumerate(m):
        try: out.append((pd.Timestamp(x.group()), text[x.start(): m[i+1].start() if i+1 < len(m) else len(text)]))
        except Exception: pass
    return out
def keep_sections(text):
    hs = list(HEADER_RE.finditer(text)); out = {}
    for i, h in enumerate(hs):
        name = h.group(1)
        if SOCIAL.search(name): continue
        body = text[h.end(): hs[i+1].start() if i+1 < len(hs) else len(text)].strip()
        for k, p in KEEP.items():
            if p.search(name): out[k] = (out.get(k, '') + ' ' + body).strip(); break
    return out

STOP = set(stopwords.words('english')) | {'were', 'was', 'conducted', 'hearing', 'examination', 'examinations',
        'employment', 'patient', 'presents', 'history', 'reports', 'denies', 'mg', 'tablet'}
PRON = re.compile(r'\b(he|she|him|her|his|hers|himself|herself|they|them|their|theirs|themselves)\b', re.I)
PHI = [(re.compile(r'\b\d{1,2}/\d{1,2}/\d{2,4}\b'), ' DATE '), (re.compile(r'\b\d{3}-\d{3}-\d{4}\b'), ' PHONE '),
       (re.compile(r'\b\d{3}-\d{2}-\d{4}\b'), ' SSN '), (re.compile(r'\b(mr|mrs|ms|dr)\.?\s+[A-Z][a-z]+\b', re.I), ' NAME ')]
AGE = re.compile(r'\b(\d{1,3})\s*(?:years?[\s-]*old|y\.?o\.?|yo)\b', re.I)
PROTECT = {'nkda', 'pcn', 'asa', 'nsaid', 'sulfa', 'date', 'phone', 'ssn', 'name', 'lab_high', 'lab_low'}

def tokenize(t):
    for p, r in PHI: t = p.sub(r, t)
    t = AGE.sub(lambda m: f" AGE_{int(m.group(1))//10*10}_{int(m.group(1))//10*10+9} " if int(m.group(1)) <= 120 else m.group(0), t)
    t = re.sub(r'\b\d+(\.\d+)?\s*\(?H\)?\b', ' LAB_HIGH ', t); t = re.sub(r'\b\d+(\.\d+)?\s*\(?L\)?\b', ' LAB_LOW ', t)
    out = []
    for w in t.split():
        w = w.strip('.,;:()[]{}*#-'); lw = w.lower()
        if not w: continue
        if w.startswith('AGE_'): out.append(w); continue
        if lw in PROTECT: out.append(lw); continue
        if lw in STOP or len(lw) < 3 or not re.search('[a-z]', lw): continue
        out.append(lw)
    return out

In [ ]:
# Cell 3 - build the 524-number vectors for each synthetic set (same steps as notebook 07, whole stay)
VITALS = {'Heart rate': 'hr', 'Systolic Blood Pressure': 'sbp', 'Diastolic Blood Pressure': 'dbp', 'Respiratory rate': 'rr', 'Body Weight': 'weight'}
CHANNELS = ['hr', 'sbp', 'dbp', 'rr', 'weight']; VCOLS = [f"vital_{c}" for c in CHANNELS]
LAB_SKIP = set(VITALS) | {'Body Height', 'Body mass index (BMI) [Ratio]', 'Pain severity - 0-10 verbal numeric rating [Score] - Reported', 'DALY', 'QOLS', 'QALY'}
def ts(s): return pd.to_datetime(s, errors='coerce', utc=True, format="ISO8601").dt.tz_localize(None)
def pool(tokens, m):
    v = [m.wv[t] for t in tokens if t in m.wv]
    return np.mean(v, 0).astype('float32') if v else np.zeros(W2V_DIM, 'float32')
def known_share(maps, m):
    toks = [t for v in maps.values() for t in v]; return np.mean([t in W2V[m].wv for t in toks]) if toks else np.nan

def build(SYN):
    OUT = f"{SYN}/features_524"; os.makedirs(OUT, exist_ok=True)
    enc = pd.read_csv(f"{SYN}/encounters.csv").rename(columns={'Id': 'hadm_id', 'PATIENT': 'subject_id', 'START': 'admittime', 'STOP': 'dischtime',
                                                           'DESCRIPTION': 'encounter_type', 'REASONDESCRIPTION': 'encounter_reason'})
    pat = pd.read_csv(f"{SYN}/patients.csv").rename(columns={'Id': 'subject_id', 'GENDER': 'gender', 'RACE': 'race', 'ETHNICITY': 'ethnicity', 'BIRTHDATE': 'birthdate'})
    df = enc.merge(pat[['subject_id', 'gender', 'race', 'ethnicity', 'birthdate']], on='subject_id', how='left')
    for c in ('admittime', 'dischtime', 'birthdate'): df[c] = ts(df[c].astype(str))
    df['target'] = (df.dischtime - df.admittime).dt.total_seconds() / 86400
    df['split'] = 'synthetic'; df['is_plos'] = (df.target > THR).astype(int)
    df['anchor_age'] = (df.admittime - df.birthdate).dt.days // 365; df['wend'] = df.dischtime
    cohort = set(df.hadm_id)
    # vitals (mean over the stay) and lab activity
    obs = pd.read_csv(f"{SYN}/observations.csv", usecols=['ENCOUNTER', 'DATE', 'DESCRIPTION', 'VALUE']).rename(columns={'ENCOUNTER': 'hadm_id'})
    obs['DATE'] = ts(obs.DATE.astype(str)); v = obs[obs.DESCRIPTION.isin(VITALS)].copy()
    v['ch'] = v.DESCRIPTION.map(VITALS); v['VALUE'] = pd.to_numeric(v.VALUE, errors='coerce')
    vit = v.groupby(['hadm_id', 'ch'])['VALUE'].mean().unstack().reindex(columns=CHANNELS); vit.columns = VCOLS
    o = obs.merge(df[['hadm_id', 'admittime', 'wend']], on='hadm_id')
    lab_ids = set(o.loc[(o.DATE >= o.admittime) & (o.DATE <= o.wend) & ~o.DESCRIPTION.isin(LAB_SKIP), 'hadm_id'])
    # medications of the stay
    med = pd.read_csv(f"{SYN}/medications.csv", usecols=['START', 'STOP', 'ENCOUNTER', 'PATIENT', 'DESCRIPTION'])
    med = med[med.ENCOUNTER.isin(cohort)].rename(columns={'ENCOUNTER': 'hadm_id'})
    med['START'] = ts(med.START.astype(str)); med['tok'] = med.DESCRIPTION.apply(med_token); med = med[med.tok != 'UNKNOWN']
    m2 = med.merge(df[['hadm_id', 'admittime', 'dischtime']], on='hadm_id')
    med_ids = set(m2.loc[(m2.START >= m2.admittime) & (m2.START <= m2.dischtime), 'hadm_id'])
    enc_meds = med.sort_values(['hadm_id', 'START']).groupby('hadm_id')['tok'].apply(list).to_dict()
    # conditions (at the stay + active at admission) and procedures
    cond = pd.read_csv(f"{SYN}/conditions.csv", usecols=['START', 'STOP', 'PATIENT', 'ENCOUNTER', 'CODE'], dtype={'CODE': str})
    for c in ('START', 'STOP'): cond[c] = ts(cond[c].astype(str))
    cond = cond.rename(columns={'PATIENT': 'subject_id', 'ENCOUNTER': 'cond_enc'})
    at_stay = cond[cond.cond_enc.isin(cohort)][['cond_enc', 'CODE', 'START']].rename(columns={'cond_enc': 'hadm_id'})
    at_stay = at_stay.merge(df[['hadm_id', 'wend']], on='hadm_id'); at_stay = at_stay[at_stay.START <= at_stay.wend][['hadm_id', 'CODE', 'START']]
    c2 = cond.merge(df[['hadm_id', 'subject_id', 'admittime']], on='subject_id')
    active = c2[(c2.START < c2.admittime) & (c2.STOP.isna() | (c2.STOP >= c2.admittime))][['hadm_id', 'CODE', 'START']]
    cev = pd.concat([at_stay, active]).drop_duplicates(['hadm_id', 'CODE']).sort_values(['hadm_id', 'START'])
    enc_cond = cev.groupby('hadm_id')['CODE'].apply(lambda s: [str(x) for x in s]).to_dict()
    proc = pd.read_csv(f"{SYN}/procedures.csv", usecols=['START', 'ENCOUNTER', 'CODE', 'DESCRIPTION'], dtype={'CODE': str}).rename(columns={'ENCOUNTER': 'hadm_id'})
    proc['START'] = ts(proc.START.astype(str)); proc = proc.sort_values(['hadm_id', 'START'])
    proc['pcode'] = proc.DESCRIPTION.map(PROC_MAP) if PROC_MAP else None
    proc['pcode'] = proc['pcode'].fillna('SCT_' + proc.CODE.astype(str))
    enc_proc = proc.groupby('hadm_id')['pcode'].apply(lambda s: [str(x) for x in s]).to_dict()
    # notes: one file per synthetic patient; match the dated segment to the stay (+/- 3 days), keep key sections, tokenize
    ndir = f"{OUT}/notes_local"
    import shutil; shutil.rmtree(ndir, ignore_errors=True)        # always unpack fresh: the notes change when the data is regenerated
    with tarfile.open(f"{SYN}/notes.tar.gz", "r:gz") as tf: tf.extractall(ndir)
    files = {f.replace('.txt', '').split('_')[-1]: os.path.join(ndir, "notes", f) for f in os.listdir(f"{ndir}/notes")}
    notes = {}
    for sid, g in df.groupby('subject_id'):
        if sid not in files: continue
        segs = segments(open(files[sid], encoding='utf-8', errors='ignore').read())
        cand = sorted((abs(sd - a), e, s) for e, a in enumerate(g.admittime) for s, (sd, _) in enumerate(segs) if abs(sd - a) <= timedelta(days=3))
        ue, us = set(), set()
        for _, e, s in cand:
            if e in ue or s in us: continue
            notes[g.hadm_id.iloc[e]] = keep_sections(segs[s][1]); ue.add(e); us.add(s)
    note_toks = {}
    for hid, secs in notes.items():
        if 'hpi' in secs: secs = {**secs, 'hpi': re.sub(r'\s+', ' ', PRON.sub('', secs['hpi']))}
        note_toks[hid] = [w for k in sorted(secs) for w in tokenize(expand(secs[k]))]
    MAPS = {'conditions': enc_cond, 'procedures': enc_proc, 'notes': note_toks, 'medications': enc_meds}
    pickle.dump({'maps': MAPS, 'notes': notes}, open(f"{OUT}/maps.pkl", 'wb'))
    # assemble exactly like notebook 07 Cell 11, with the REAL category codes and REAL Word2Vec models
    X = df[['hadm_id', 'subject_id', 'split', 'admittime', 'target', 'is_plos', 'anchor_age', 'gender', 'race', 'ethnicity',
            'encounter_type', 'encounter_reason']].merge(vit, left_on='hadm_id', right_index=True, how='left')
    X['has_vitals'] = X[VCOLS].notna().any(axis=1).astype('int8'); X[VCOLS] = X[VCOLS].fillna(0)
    X['gender_enc'] = (X.gender == 'M').astype('int8')
    X['race_enc'] = X.race.map(RACE).fillna(-1).astype(int); X['ethnicity_enc'] = X.ethnicity.map(ETH).fillna(-1).astype(int)
    X['lab_active'] = X.hadm_id.isin(lab_ids).astype('int8'); X['med_active'] = X.hadm_id.isin(med_ids).astype('int8')
    for name in MODS:
        E = np.stack([pool(MAPS[name].get(h, []), W2V[name]) for h in X.hadm_id])
        X = pd.concat([X, pd.DataFrame(E, columns=[f"{name[:4]}_{i}" for i in range(W2V_DIM)], index=X.index)], axis=1)
    assert [c for c in FM["feat524"] if c not in X.columns] == []
    X.to_parquet(f"{OUT}/encounters_524.parquet", index=False)
    seqs = {sid: {'X': g[FM["feat524"]].values.astype('float32'), 'Y': g.target.values.astype('float32'), 'hadm_ids': g.hadm_id.tolist(),
                  'split': 'synthetic', 'n_encounters': len(g)} for sid, g in X.sort_values('admittime').groupby('subject_id')}
    pickle.dump(seqs, open(f"{OUT}/sequences_patient_level.pkl", "wb"))
    json.dump({**FM, "source": SYN, "word2vec_from": REAL}, open(f"{OUT}/features_meta.json", "w"), indent=1)
    known = {m: known_share(MAPS[m], m) for m in MODS}
    return X, known, len(notes)

XS, KNOWN = {}, {}
for name, path in SETS.items():
    if not os.path.exists(f"{path}/encounters.csv"): print(f"{name}: no files at {path}, skipped"); continue
    XS[name], KNOWN[name], n_notes = build(path)
    print(f"{name:10s}: {len(XS[name]):,} stays | notes matched {n_notes:,} | PLOS {XS[name].is_plos.mean():.0%} | saved {path}/features_524/")

In [ ]:
# Cell 4 - coverage: modality present per stay (real PLOS vs synthetic) and share of tokens the real Word2Vec knows
F = FM["feat524"]; BLK = FM["blocks"]
def presence(X):
    out = {f"has_{m}": (X[BLK[m]].abs().sum(1) > 0).mean() for m in MODS}
    out.update({"has_vitals": X.has_vitals.mean(), "lab_active": X.lab_active.mean(), "med_active": X.med_active.mean()}); return out
RP = XR[XR.is_plos == 1]
cov = pd.DataFrame({"real PLOS (all splits)": presence(RP), **{f"synthetic {k}": presence(v) for k, v in XS.items()}})
print("Share of stays where each modality is present:"); print(cov.round(3).to_string())
print("\nShare of synthetic tokens found in the real Word2Vec vocabulary:"); print(pd.DataFrame(KNOWN).round(3).to_string())

In [ ]:
# Cell 5 - real vs synthetic in the 524-space: discriminator AUC (all 524 and per block), with a real-vs-real reference
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
def disc(A, B, cols, ga, gb, seed=0):
    X = pd.concat([A[cols], B[cols]], ignore_index=True).values; y = np.r_[np.zeros(len(A)), np.ones(len(B))]
    g = np.r_[np.asarray(ga, dtype=str), np.asarray(gb, dtype=str)]; p = np.zeros(len(y))
    for tr, te in StratifiedGroupKFold(5, shuffle=True, random_state=seed).split(X, y, g):
        p[te] = HistGradientBoostingClassifier(max_iter=200, random_state=seed).fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]
    return roc_auc_score(y, p)
HO = XR[(XR.is_plos == 1) & XR.split.isin(["val", "test"])].reset_index(drop=True)      # held-out real PLOS
TR = XR[(XR.is_plos == 1) & (XR.split == "train")].reset_index(drop=True)
PROFILE = FM["profile"]
COLSETS = {"all 524": F, "profile + vitals": PROFILE, **{m: BLK[m] for m in MODS}, "flags (lab, med)": ["lab_active", "med_active"]}
rows = []
rng = np.random.default_rng(0)
for cname, cols in COLSETS.items():
    r = {"features": cname}
    TRs = TR.sample(min(len(TR), len(HO)), random_state=0)
    r["real train vs real held-out (reference)"] = disc(HO, TRs, cols, HO.subject_id, TRs.subject_id)
    for k, X in XS.items():
        Xs = X.sample(min(len(X), len(HO)), random_state=0)
        r[f"synthetic {k} vs real held-out"] = disc(HO, Xs, cols, HO.subject_id, Xs.subject_id)
    rows.append(r)
D = pd.DataFrame(rows).set_index("features"); D.to_csv(f"{BASE}/synthetic_full/results_13_discriminator.csv")
print(f"Held-out real PLOS {len(HO)} | training real PLOS {len(TR)} (0.5 = indistinguishable)")
D.round(3)

In [ ]:
# Cell 6 - would a model trained on REAL data call these long stays? (classifier on real features_full, train split)
from sklearn.metrics import average_precision_score
tr, va = XR[XR.split == "train"], XR[XR.split.isin(["val", "test"])]
clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=0).fit(tr[F], tr.is_plos)
pv = clf.predict_proba(va[F])[:, 1]
print(f"Classifier on real data: held-out AP {average_precision_score(va.is_plos, pv):.3f} (prevalence {va.is_plos.mean():.3f})")
thr = np.quantile(pv[va.is_plos.values == 1], 0.5)        # threshold that catches half of real held-out PLOS stays
def score(p): return {"mean P(PLOS)": p.mean(), "median P(PLOS)": np.median(p), "share above threshold": (p >= thr).mean()}
S6 = {"real held-out PLOS": score(pv[va.is_plos.values == 1]), "real held-out non-PLOS": score(pv[va.is_plos.values == 0])}
for k, X in XS.items():
    ps = clf.predict_proba(X[F])[:, 1]; S6[f"synthetic {k}"] = score(ps)
    if k == "tail_dense": S6["synthetic tail_dense, extreme stays only"] = score(ps[(X.target > np.quantile(TR.target, 0.9)).values])
print(f"Threshold = median score of real held-out PLOS stays ({thr:.3f}); by construction 50% of them are above it")
print(f"lab_active among PLOS: real {RP.lab_active.mean():.2f} | " + ", ".join(f"synthetic {k} {X.lab_active.mean():.2f}" for k, X in XS.items()))
print(pd.DataFrame(S6).T.round(3).to_string())
# same check without lab_active, so the missing labs do not drive the answer
F2 = [c for c in F if c != "lab_active"]
clf2 = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=0).fit(tr[F2], tr.is_plos)
pv2 = clf2.predict_proba(va[F2])[:, 1]; thr = np.quantile(pv2[va.is_plos.values == 1], 0.5)
S7 = {"real held-out PLOS": score(pv2[va.is_plos.values == 1]), "real held-out non-PLOS": score(pv2[va.is_plos.values == 0]),
      **{f"synthetic {k}": score(clf2.predict_proba(X[F2])[:, 1]) for k, X in XS.items()}}
print(f"\nWithout lab_active (held-out AP {average_precision_score(va.is_plos, pv2):.3f}):")
print(pd.DataFrame(S7).T.round(3).to_string())

In [ ]:
# Cell 7 - diagnostics: what differs between real and synthetic token lists (the input of each Word2Vec block)?
import collections
RF = REAL
rn = pickle.load(open(f"{RF}/encounter_notes.pkl", "rb")); rt = pickle.load(open(f"{RF}/encounter_notes_tokens.pkl", "rb"))
rm = pickle.load(open(f"{RF}/meds.pkl", "rb"))["enc_meds"]; cp = pickle.load(open(f"{RF}/cond_proc.pkl", "rb"))
r_cond = cp["cevents"].groupby("hadm_id")["CODE"].apply(lambda s: [str(x) for x in s]).to_dict()
pr = cp["proc"]; pr = pr.assign(pcode=(pr.DESCRIPTION.map(PROC_MAP) if PROC_MAP else None)); pr["pcode"] = pr["pcode"].fillna("SCT_" + pr.CODE.astype(str))
r_proc = pr.groupby("hadm_id")["pcode"].apply(lambda s: [str(x) for x in s]).to_dict()
REALMAPS = {"conditions": r_cond, "procedures": r_proc, "notes": rt, "medications": rm}
ids = list(HO.hadm_id)                                         # held-out real PLOS stays
name = next(iter(XS)); SM = pickle.load(open(f"{SETS[name]}/features_524/maps.pkl", "rb"))
sids = list(XS[name].hadm_id)
rows = []
for m in MODS:
    R_ = [REALMAPS[m].get(h, []) for h in ids]; S_ = [SM["maps"][m].get(h, []) for h in sids]
    f = lambda L: (np.mean([len(x) for x in L if x]), np.mean([len(set(x)) for x in L if x]), np.mean([len(x) > 0 for x in L]))
    a, b = f(R_), f(S_)
    rows.append({"block": m, "real tokens/stay": a[0], "synthetic tokens/stay": b[0], "real distinct/stay": a[1],
                 "synthetic distinct/stay": b[1], "real present": a[2], "synthetic present": b[2]})
print("Token lists per stay (stays with the block present):"); print(pd.DataFrame(rows).set_index("block").round(2).to_string())
for m in ["notes", "conditions", "medications", "procedures"]:
    R_ = [set(REALMAPS[m].get(h, [])) for h in ids]; S_ = [set(SM["maps"][m].get(h, [])) for h in sids]
    dr = collections.Counter(t for x in R_ for t in x); ds = collections.Counter(t for x in S_ for t in x)
    d = pd.DataFrame({"real share": pd.Series(dr) / len(R_), "synthetic share": pd.Series(ds) / len(S_)}).fillna(0)
    d["diff"] = d["synthetic share"] - d["real share"]
    print(f"\n{m}: tokens most over-used (+) and under-used (-) in synthetic stays, share of stays containing the token")
    print(pd.concat([d.sort_values("diff").head(10), d.sort_values("diff").tail(10)]).round(3).to_string())
h = ids[int(np.argmax([len(rt.get(i, [])) for i in ids]) if False else 0)]
long_real = sorted(ids, key=lambda i: len(" ".join(rn.get(i, {}).values())))[len(ids) // 2]
long_syn = sorted(sids, key=lambda i: len(" ".join(SM["notes"].get(i, {}).values())))[len(sids) // 2]
for lab, secs in (("REAL held-out PLOS stay (kept sections)", rn.get(long_real, {})), ("SYNTHETIC stay (kept sections)", SM["notes"].get(long_syn, {}))):
    print(f"\n=============== {lab} ===============")
    for k, v in secs.items(): print(f"--- {k} ---\n{v[:1200]}")